In [3]:
import torch
import numpy as np

In [9]:
x = torch.tensor([1,2,3,4])
x

tensor([1, 2, 3, 4])

In [7]:
y = np.array([1,2,3,4])
y

array([1, 2, 3, 4])

In [11]:
y.sum()

10

In [13]:
x.sum()

tensor(10)

In [17]:
X = torch.ones([4,4])
X

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [19]:
Y = torch.randn([4,4])
Y

tensor([[ 0.5253,  2.3068, -0.0176, -1.4089],
        [ 1.9448,  1.1378, -0.7707, -0.8052],
        [-0.0671, -0.7399, -0.6981,  1.6812],
        [ 2.1088,  0.3171,  0.7172, -1.8117]])

In [23]:
X+Y

tensor([[ 1.5253,  3.3068,  0.9824, -0.4089],
        [ 2.9448,  2.1378,  0.2293,  0.1948],
        [ 0.9329,  0.2601,  0.3019,  2.6812],
        [ 3.1088,  1.3171,  1.7172, -0.8117]])

In [25]:
Y @ X

tensor([[1.4056, 1.4056, 1.4056, 1.4056],
        [1.5067, 1.5067, 1.5067, 1.5067],
        [0.1761, 0.1761, 0.1761, 0.1761],
        [1.3315, 1.3315, 1.3315, 1.3315]])

# задача 1

In [34]:
a = list(range(1, 10**4))

In [60]:
%timeit sum([i**2 for i in a])

503 μs ± 45 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [62]:
a_n = np.array(a, dtype=np.int64)
%timeit (a_n**2).sum()

8.04 μs ± 259 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [64]:
a_t = torch.tensor(a)
a_t

tensor([   1,    2,    3,  ..., 9997, 9998, 9999])

In [106]:
%timeit (a_t ** 2).sum()

22.5 μs ± 566 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [110]:
x = torch.empty([5,3])
x

tensor([[1.7136e+35, 2.1076e-42, 6.1041e-42],
        [0.0000e+00, 6.2904e-42, 0.0000e+00],
        [6.4796e-42, 0.0000e+00, 6.6716e-42],
        [0.0000e+00, 6.8664e-42, 0.0000e+00],
        [7.0639e-42, 0.0000e+00, 7.2643e-42]])

# Градиенты

In [70]:
x = torch.randn([4,4], requires_grad=True)
y = torch.randn([4,4], requires_grad=True)

In [76]:
l = torch.sum((x + y)**2)

In [78]:
print(x.grad)

None


In [80]:
l.backward()

In [82]:
x.grad

tensor([[ 4.0246,  5.7575, -0.1750, -3.4595],
        [-2.7933, -1.5577, -3.5008, -0.7684],
        [-0.5578,  0.7565, -0.5530, -1.5547],
        [-5.9759,  3.2659, -2.0958, -1.0790]])

In [94]:
print(2 * (x[0, 0] + y[0, 0]))
print(x.grad[0, 0])

tensor(4.0246, grad_fn=<MulBackward0>)
tensor(4.0246)


In [104]:
x.grad.zero_()

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

### Задание 2:

Реализуйте на PyTorch сигмоиду 

$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

In [117]:
import math
def sigm(x):
    res = 1 / (1 + math.exp**(-x))
    return res

### Задание 3:

Реализуйте на PyTorch среднюю квадратичную ошибку. 

$$ 
MSE(\hat y, y) = \frac{1}{n} \cdot \sum_{i=1}^n (\hat y - y)^2
$$

In [121]:
def mse(y_true, y_pred):
    res = torch.mean((y_true - y_pred) ** 2)
    return res

### Задание 4:

Что будет в переменной `x` в результате выполнения следующего кода? Почему?

In [124]:
import torch
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
y = x
y[2] = torch.ones(3)

In [126]:
x

tensor([[1, 2, 3],
        [4, 5, 6],
        [1, 1, 1]])

In [130]:
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
y = x.clone()
y[2] = torch.ones(3)

In [132]:
x

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

# 2. Производные да градиенты

Если в PyTorch те же самые операции, что и в numpy, то на кой чёрт он нам нужен? В отличие от numpy, в PyTorch есть возможность при создании тензора указывать нужно ли считать по нему градиент или нет, с помощью параметра `requires_grad`. 

Когда `requires_grad=True` мы сообщаем фреймворку, о том, что мы хотим следить за всеми тензорами, которые получаются из созданного. Иными словами, у любого тензора, у которого указан данный параметр, будет доступ к цепочке операций и преобразований совершенными с ними. Если эти функции дифференцируемые, то у тензора появляется параметр `.grad`, в котором хранится значение градиента.

Он проходит по всем операциям, которые фигурируют в графе вычислений, и применяет к ним chain rule:

$$ {\partial f(g(x)) \over \partial x} = {\partial f(g(x)) \over \partial g(x)}\cdot {\partial g(x) \over \partial x} $$

Давайте попробуем!

In [143]:
x = torch.tensor([0.3, 1], requires_grad=True)
y = torch.tensor([0.1, 2], requires_grad=True)

z = sum((x + y) ** 2)

In [145]:
print(x.grad)

None


In [147]:
z.backward()

In [149]:
x.grad

tensor([0.8000, 6.0000])

In [151]:
y.grad

tensor([0.8000, 6.0000])

### Задание 5:

Реализуйте расчёт градиента для функции 

$$
f(w) = \prod_{i,j} \ln(\ln(w_{ij} + 7) 
$$

в точке `w = [[5,10], [1,2]]`

In [168]:
w = torch.tensor([[5.0, 10.0], [1.0, 2.0]], requires_grad=True)

In [176]:
f = torch.prod(torch.log(torch.log(w + 7)))
f

tensor(0.5463, grad_fn=<ProdBackward0>)

In [174]:
print(w.grad)

None


In [178]:
f.backward()

In [180]:
print(w.grad)

tensor([[0.0201, 0.0109],
        [0.0449, 0.0351]])


### Задание 6:

Реализуйте для функции 

$$
f(w) = \prod_{i,j} \ln(\ln(w_{ij} + 7) 
$$

процедуру градиентного спуска. Каким получилось минимальное значение? 

Релиазовал в 5